# Notebook 01 — Exploração do Modelo Base e Análise de Pesos

## Aula 04: Otimização de Modelos — Pruning e Quantização

---

### 🎯 Objetivos deste Notebook

1. Carregar o dataset CIFAR-10 e realizar EDA visual
2. Definir e inspecionar a arquitetura da CNN base (`SimpleCNN`)
3. Analisar a **distribuição de pesos** da rede — base para entender pruning
4. Visualizar a **redundância paramétrica** conforme Denil et al. (2013)
5. Introduzir conceitos da **Hipótese da Lottery Ticket** (Frankle & Carlin, 2019)

### 📖 Conceito Teórico (Documento 04 — Saiba Mais)

> A redundância em redes neurais profundas foi identificada empiricamente por
> Denil et al. (2013), que demonstraram que os pesos de uma rede podem ser
> preditos com alta precisão a partir de um subconjunto reduzido, sugerindo que
> a maior parte dos parâmetros carrega informação redundante.

> A **Lottery Ticket Hypothesis** (Frankle & Carlin, 2019) postula que uma rede
> neural densa contém uma sub-rede esparsa (*winning ticket*) que, treinada
> desde a mesma inicialização, atinge acurácia comparável à rede completa.

---

In [ ]:
# Imports necessários
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn

# Adicionar diretório raiz ao path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import DataPreprocessor, CIFAR10_CLASSES
from src.model import SimpleCNN, StudentCNN, count_parameters, get_model_size_mb
from src.utils import set_seed, get_device

# Configurações de visualização
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Reprodutibilidade
set_seed(42)
device = get_device()
print(f"Dispositivo: {device}")

## 1. Carregamento do CIFAR-10

O **CIFAR-10** (Krizhevsky, 2009) contém 60 000 imagens 32×32 RGB em 10 classes.
Utilizamos `torchvision` para download automático e normalização com estatísticas
padrão (μ e σ por canal).

In [ ]:
# Carregar CIFAR-10 via DataPreprocessor
preprocessor = DataPreprocessor(data_dir="data/raw", batch_size=128, seed=42)
preprocessor.load_data(download=True)
train_loader, val_loader, test_loader = preprocessor.split_data(val_fraction=0.1)

print(f"Batches de treino : {len(train_loader)}")
print(f"Batches de valid. : {len(val_loader)}")
print(f"Batches de teste  : {len(test_loader)}")

## 2. Visualização de Amostras

Exibimos amostras de cada classe para ter intuição visual sobre os dados.

In [ ]:
# Desnormalizar para visualização
def denormalize(tensor, mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)):
    """Desfaz normalização para exibir imagem."""
    t = tensor.clone()
    for ch, m, s in zip(t, mean, std):
        ch.mul_(s).add_(m)
    return t.clamp(0, 1)

# Obter um batch
images, labels = next(iter(test_loader))

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CIFAR10_CLASSES[labels[i].item()], fontsize=10)
    ax.axis("off")
plt.suptitle("Amostras do CIFAR-10", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Distribuição de Classes

Verificamos o balanceamento das classes — 5 000 amostras/classe no treino.

In [ ]:
# Contagem de classes no dataset de treino completo
from torchvision import datasets
train_full = datasets.CIFAR10(root="data/raw", train=True, download=False)
class_counts = np.bincount([label for _, label in train_full])

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(CIFAR10_CLASSES, class_counts, color=sns.color_palette("husl", 10))
ax.set_ylabel("Amostras")
ax.set_title("Distribuição de Classes — CIFAR-10 (treino)")
for bar, count in zip(bars, class_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Arquitetura do Modelo Base — SimpleCNN

A CNN base possui ~1.1 M parâmetros. Conforme discutido no doc 04, modelos
sobreparametrizados facilitam a otimização, mas a maioria dos parâmetros é
redundante (Denil et al., 2013).

In [ ]:
# Instanciar modelo base
model = SimpleCNN(num_classes=10)

# Estatísticas do modelo
n_params = count_parameters(model)
size_mb = get_model_size_mb(model)

print(f"{'='*50}")
print(f"  SimpleCNN — Modelo Base")
print(f"{'='*50}")
print(f"  Parâmetros treináveis : {n_params:,}")
print(f"  Tamanho em memória   : {size_mb:.2f} MB")
print(f"{'='*50}")
print()
print(model)

## 5. Comparação: Teacher vs. Student

Conforme Hinton et al. (2015), o modelo *student* deve ser substancialmente
menor que o *teacher* para que Knowledge Distillation faça sentido.

In [ ]:
student = StudentCNN(num_classes=10)
n_params_student = count_parameters(student)
size_student = get_model_size_mb(student)

print(f"{'Model':<15} {'Params':>12} {'Size (MB)':>10} {'Ratio':>8}")
print(f"{'-'*50}")
print(f"{'SimpleCNN':<15} {n_params:>12,} {size_mb:>10.2f} {'1.0x':>8}")
print(f"{'StudentCNN':<15} {n_params_student:>12,} {size_student:>10.2f} {n_params/n_params_student:>7.1f}x")

## 6. Análise da Distribuição de Pesos (antes do treinamento)

Antes do treinamento, os pesos seguem a inicialização padrão do PyTorch
(Kaiming uniform para Conv2d, etc.). A maioria dos valores concentra-se
perto de zero — este é o fundamento do *magnitude-based pruning* de
Han et al. (2015).

In [ ]:
# Distribuição de pesos por camada (modelo não treinado)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes_flat = axes.flatten()

idx = 0
for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)) and idx < 6:
        w = module.weight.data.cpu().numpy().flatten()
        axes_flat[idx].hist(w, bins=80, alpha=0.7, color="steelblue", edgecolor="none")
        axes_flat[idx].set_title(f"{name}\n(shape: {list(module.weight.shape)})", fontsize=9)
        axes_flat[idx].set_xlabel("Peso")
        axes_flat[idx].axvline(0, color="red", linestyle="--", alpha=0.5)
        pct_near_zero = 100.0 * np.sum(np.abs(w) < 0.01) / w.size
        axes_flat[idx].text(0.95, 0.95, f"|w|<0.01: {pct_near_zero:.1f}%",
                            transform=axes_flat[idx].transAxes, ha="right", va="top", fontsize=8)
        idx += 1

for ax in axes_flat[idx:]:
    ax.set_visible(False)

plt.suptitle("Distribuição de Pesos — SimpleCNN (não treinado)", fontsize=13)
plt.tight_layout()
plt.savefig("outputs/figures/weight_dist_untrained.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Contagem de Parâmetros por Camada

Identificamos quais camadas concentram mais parâmetros — candidatas
prioritárias para pruning estruturado (doc 04, Tabela 1).

In [ ]:
# Parâmetros por camada
layer_info = []
for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        n = module.weight.nelement()
        layer_info.append({"Camada": name, "Parâmetros": n, "Shape": str(list(module.weight.shape))})

import pandas as pd
df_layers = pd.DataFrame(layer_info)
df_layers["% do Total"] = 100.0 * df_layers["Parâmetros"] / df_layers["Parâmetros"].sum()
print(df_layers.to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(df_layers["Camada"], df_layers["% do Total"], color="steelblue")
ax.set_xlabel("% dos Parâmetros Totais")
ax.set_title("Distribuição de Parâmetros por Camada — SimpleCNN")
plt.tight_layout()
plt.savefig("outputs/figures/params_per_layer.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Hipótese da Lottery Ticket — Intuição Visual

Conforme Frankle & Carlin (2019), se sortearmos uma máscara aleatória
que zere grande parte dos pesos, o modelo perderá capacidade.
Porém, a **máscara correta** (winning ticket) preserva a acurácia.

Simularemos esparsidade aplicando limiar de magnitude nos pesos iniciais
para ver quantos estão próximos de zero.

In [ ]:
# Simular diferentes níveis de esparsidade por limiar de magnitude
all_weights = torch.cat([p.data.flatten() for p in model.parameters()])
magnitudes = all_weights.abs().numpy()

thresholds = np.percentile(magnitudes, [10, 30, 50, 70, 90])
print("Percentis de magnitude dos pesos:")
for pct, thr in zip([10, 30, 50, 70, 90], thresholds):
    n_below = np.sum(magnitudes < thr)
    print(f"  P{pct:2d}: limiar = {thr:.4f}  →  {n_below:,} pesos removidos "
          f"({100*n_below/len(magnitudes):.1f}%)")

# Histograma de magnitudes globais
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(magnitudes, bins=150, alpha=0.7, color="steelblue", edgecolor="none")
for thr, pct in zip(thresholds, [10, 30, 50, 70, 90]):
    ax.axvline(thr, color="red", linestyle="--", alpha=0.6)
    ax.text(thr, ax.get_ylim()[1]*0.9, f"P{pct}", fontsize=8, ha="center")
ax.set_xlabel("|w|")
ax.set_ylabel("Frequência")
ax.set_title("Distribuição de |peso| — SimpleCNN (Lottery Ticket Intuition)")
plt.tight_layout()
plt.savefig("outputs/figures/magnitude_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Contagem: FLOPs Estimados por Camada

A quantidade de FLOPs (operações de ponto flutuante) impacta diretamente
a latência de inferência. O pruning estruturado reduz FLOPs de forma
efetiva (doc 04, Tabela 1).

In [ ]:
# Estimativa simplificada de FLOPs por camada
def estimate_flops_conv2d(module: nn.Conv2d, input_h: int, input_w: int) -> int:
    """FLOPs ≈ 2 × C_in × K_h × K_w × C_out × H_out × W_out"""
    c_out, c_in, kh, kw = module.weight.shape
    h_out = (input_h + 2 * module.padding[0] - kh) // module.stride[0] + 1
    w_out = (input_w + 2 * module.padding[1] - kw) // module.stride[1] + 1
    return 2 * c_in * kh * kw * c_out * h_out * w_out

def estimate_flops_linear(module: nn.Linear) -> int:
    """FLOPs ≈ 2 × in_features × out_features"""
    return 2 * module.in_features * module.out_features

# Percorrer camadas com tamanhos de feature map conhecidos
flops_info = []
h, w = 32, 32
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        f = estimate_flops_conv2d(module, h, w)
        flops_info.append({"Camada": name, "FLOPs": f})
        h_out = (h + 2*module.padding[0] - module.kernel_size[0]) // module.stride[0] + 1
        w_out = (w + 2*module.padding[1] - module.kernel_size[1]) // module.stride[1] + 1
        h, w = h_out, w_out
    elif isinstance(module, nn.MaxPool2d):
        h, w = h // 2, w // 2
    elif isinstance(module, nn.Linear):
        f = estimate_flops_linear(module)
        flops_info.append({"Camada": name, "FLOPs": f})

df_flops = pd.DataFrame(flops_info)
df_flops["MFLOPs"] = df_flops["FLOPs"] / 1e6
df_flops["% Total"] = 100 * df_flops["FLOPs"] / df_flops["FLOPs"].sum()
print(df_flops.to_string(index=False))
print(f"\nTotal: {df_flops['FLOPs'].sum()/1e6:.1f} MFLOPs")

## 10. Resumo e Próximos Passos

Neste notebook exploramos:

- **Dataset CIFAR-10**: 50k imagens de treino, 10k teste, 10 classes balanceadas
- **Modelo base** (SimpleCNN): ~1.1M parâmetros, ~4 MB FP32
- **Distribuição de pesos**: concentrada perto de zero → base para pruning
- **Redundância**: camadas FC concentram a maioria dos parâmetros
- **Lottery Ticket**: grande fração dos pesos pode ser zerda sem impacto

### ➡️ Próximo Notebook: `02_treinamento.ipynb`

No próximo notebook, treinaremos a SimpleCNN, aplicaremos **pruning** (Han et al.,
2015), **quantização INT8** (Jacob et al., 2018) e **Knowledge Distillation**
(Hinton et al., 2015) conforme o pipeline integrado do documento 04.